In [0]:
# =============================================================================
# NB10 — Full vs selected feature sets, with in-fold selection + temporal test
# =============================================================================
# Objective:
#   1. Compare FULL vs SELECTED feature sets for L1 / L2 / L3 under the
#      serialized main_cv splits from NB07 (GroupKFold by DOI).
#      Feature selection is repeated INSIDE each fold (same procedure as NB09:
#      RF-MDI ranking -> |r|>0.90 correlation filter -> top-N -> physical
#      must-keeps), using only that fold's training rows. This avoids the
#      selection leakage that would occur if the global NB09 list were reused
#      (NB09 selected on fold-0 train, which contains val rows of folds 1-4).
#   2. y-randomization sanity check on the best configuration.
#   3. Temporal hold-out (most recent 20% of DOIs, from NB07) for the best
#      configuration: train on older papers, test on newer ones.
#
#   NOTE: RF OOB R² is NOT a valid metric here — bootstrap is row-level and
#   leaks intra-paper information. Only grouped CV / temporal hold-out are
#   reported.
#
# Inputs:
#   ml_perovskites.master, cv_splits, y_randomized,
#   X_material, X_thermo, X_exp
#
# Outputs:
#   ml_perovskites.cv_results_nb10      (fold × model × layer × feature_set)
#   ml_perovskites.temporal_results_nb10
#   /tmp/nb10_full_vs_selected.png
#
# Rules: imputation, target encoding and feature selection strictly inside
#        each fold. source_row_id / doi / doi_key never features. SEED = 42.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import copy
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute   import SimpleImputer
from sklearn.metrics  import r2_score, mean_squared_error, mean_absolute_error
from pyspark.sql import SparkSession

SEED        = 42
SCHEMA      = "ml_perovskites"
CORR_THRESH = 0.90
TOP_N       = {"L1": 40, "L2": 40, "L3": 50}
TE_M        = 10
CAT_COLS    = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type"]
MUST_KEEP_PATTERNS = [
    "goldschmidt", "octahedral", "Ra_Rb_ratio",
    "ECB_butler", "EVB_butler", "chi_mulliken",
    "co_catalyst", "scavenger", "run_type", "lamp_type",
    "catalyst_amount",
]

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

def try_import(pkg):
    try:
        return __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
        return __import__(pkg)

xgb = try_import("xgboost")
try:
    cb = try_import("catboost")
    HAS_CB = True
except Exception:
    HAS_CB = False
    print("WARNING: catboost unavailable — skipped.")

# ---------------------------------------------------------------------------
# 1. Load and align
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
cv_splits = spark.table(f"{SCHEMA}.cv_splits").toPandas()
y_rand_df = spark.table(f"{SCHEMA}.y_randomized").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp").toPandas()

def is_use_h2(series):
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
df_base = master[
    is_use_h2(master["use_h2"]) &
    master["y_log10_1p_final"].notna() &
    master["doi_key"].notna()
].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

# Join key must be the same dtype everywhere
for t in [df_base, cv_splits, y_rand_df, X_mat_raw, X_thm_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META_COLS = {"source_row_id", "base_compound", "family_proposed",
             "formula_base_paper", "A_site_paper", "B_site_paper",
             "doi", "doi_key", "y_log10_1p_final", "y_log10_1p",
             "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}

for raw in [X_mat_raw, X_thm_raw]:
    for c in raw.columns:
        if c != "source_row_id":
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

def numeric_features(d):
    return [c for c in d.columns
            if c not in META_COLS and pd.api.types.is_numeric_dtype(d[c])]

x_mat_cols = numeric_features(X_mat_raw)
x_mat_cols = [c for c in x_mat_cols if X_mat_raw[c].isna().mean() < 0.50]
x_thm_cols = numeric_features(X_thm_raw)
x_thm_cols = [c for c in x_thm_cols if X_thm_raw[c].isna().mean() < 0.99]
x_exp_num  = [c for c in X_exp_raw.columns
              if c not in ["source_row_id"] + CAT_COLS]
for c in x_exp_num:
    X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")
x_exp_num  = [c for c in x_exp_num if X_exp_raw[c].notna().any()]

df = (df_base[["source_row_id", "doi_key", "y"]]
      .merge(X_mat_raw[["source_row_id"] + x_mat_cols], on="source_row_id", how="left")
      .merge(X_thm_raw[["source_row_id"] + x_thm_cols], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + x_exp_num + CAT_COLS],
             on="source_row_id", how="left"))
df = df.merge(y_rand_df[["source_row_id", "y_rand"]], on="source_row_id", how="left")
print(f"Merged shape: {df.shape}")

LAYERS = {
    "L1": x_mat_cols,
    "L2": x_mat_cols + x_thm_cols,
    "L3": x_mat_cols + x_thm_cols + x_exp_num + CAT_COLS,   # cats encoded in fold
}
for k, v in LAYERS.items():
    print(f"  {k}: {len(v)} candidate features")

# ---------------------------------------------------------------------------
# 2. In-fold helpers
# ---------------------------------------------------------------------------
def encode_fold(tr, te, cols, ycol):
    """Smoothed target encoding fitted on tr only; returns float frames."""
    Xtr, Xte = tr[cols].copy(), te[cols].copy()
    for cc in [c for c in CAT_COLS if c in cols]:
        gm = tr[ycol].mean()
        st = tr.groupby(cc)[ycol].agg(["mean", "count"])
        enc = ((st["mean"] * st["count"] + gm * TE_M) / (st["count"] + TE_M)).to_dict()
        Xtr[cc] = tr[cc].map(enc).fillna(gm)
        Xte[cc] = te[cc].map(enc).fillna(gm)
    return Xtr.astype(float), Xte.astype(float)

def select_in_fold(Xtr_df, ytr, layer):
    """NB09 procedure on training rows only: MDI rank -> corr filter -> top-N."""
    cols = list(Xtr_df.columns)
    X_imp = SimpleImputer(strategy="median").fit_transform(Xtr_df.values)
    rf = RandomForestRegressor(n_estimators=300, max_features="sqrt",
                               min_samples_leaf=3, n_jobs=-1, random_state=SEED)
    rf.fit(X_imp, ytr)
    ranked = [c for _, c in sorted(zip(rf.feature_importances_, cols), reverse=True)]
    corr = pd.DataFrame(X_imp, columns=cols).corr().abs()
    keep = []
    for f in ranked:
        if keep and corr.loc[f, keep].max() > CORR_THRESH:
            continue
        keep.append(f)
    top = set(ranked[:TOP_N[layer]])
    sel = [c for c in keep if c in top]
    must = [c for c in cols if any(k in c for k in MUST_KEEP_PATTERNS) and c not in sel]
    return list(dict.fromkeys(sel + must))

def make_models():
    m = {
        "RF": RandomForestRegressor(n_estimators=300, max_features="sqrt",
                                    min_samples_leaf=3, n_jobs=-1, random_state=SEED),
        "XGBoost": xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                                    subsample=0.8, colsample_bytree=0.7,
                                    min_child_weight=3, reg_lambda=1.0,
                                    tree_method="hist", random_state=SEED, verbosity=0),
    }
    if HAS_CB:
        m["CatBoost"] = cb.CatBoostRegressor(iterations=600, learning_rate=0.05,
                                             depth=6, l2_leaf_reg=3,
                                             random_seed=SEED, verbose=0)
    return m

def metrics(y_true, y_pred):
    yo_t, yo_p = 10 ** y_true - 1, 10 ** y_pred - 1
    return {
        "r2_log":   r2_score(y_true, y_pred),
        "rmse_log": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_log":  mean_absolute_error(y_true, y_pred),
        "r2_orig":  r2_score(yo_t, yo_p),
    }

def fit_predict(tr, te, cols, layer, feature_set, ycol, model_name, model):
    Xtr, Xte = encode_fold(tr, te, cols, ycol)
    if feature_set == "selected":
        sel = select_in_fold(Xtr, tr[ycol].values, layer)
        Xtr, Xte = Xtr[sel], Xte[sel]
    imp = SimpleImputer(strategy="median")
    Xtr_a = imp.fit_transform(Xtr.values)
    Xte_a = imp.transform(Xte.values)
    mdl = copy.deepcopy(model)
    mdl.fit(Xtr_a, tr[ycol].values)
    return mdl.predict(Xte_a), Xtr.shape[1]

# ---------------------------------------------------------------------------
# 3. Grouped CV: layer × feature_set × model (main_cv from NB07)
# ---------------------------------------------------------------------------
main_cv = cv_splits[cv_splits["split_type"] == "main_cv"]
fold_ids = sorted(main_cv["fold_id"].unique())

def fold_frames(fold_id):
    f = main_cv[main_cv["fold_id"] == fold_id]
    tr_ids = set(f[f["role"] == "train"]["source_row_id"])
    va_ids = set(f[f["role"] == "val"]["source_row_id"])
    return df[df["source_row_id"].isin(tr_ids)], df[df["source_row_id"].isin(va_ids)]

FOLDS = {fid: fold_frames(fid) for fid in fold_ids}

rows = []
print("\n--- Grouped CV (main_cv): full vs selected (in-fold) ---")
for layer, cols in LAYERS.items():
    for fs in ["full", "selected"]:
        for mname, model in make_models().items():
            for fid, (tr, va) in FOLDS.items():
                yp, nfeat = fit_predict(tr, va, cols, layer, fs, "y", mname, model)
                r = metrics(va["y"].values, yp)
                r.update({"layer": layer, "feature_set": fs, "model": mname,
                          "fold_id": int(fid), "n_features": int(nfeat),
                          "n_train": len(tr), "n_val": len(va)})
                rows.append(r)
            sub = [x for x in rows if x["layer"] == layer and
                   x["feature_set"] == fs and x["model"] == mname]
            r2s = [x["r2_log"] for x in sub]
            print(f"  {mname:9s} | {layer} | {fs:8s} | "
                  f"R²(log)={np.mean(r2s):+.3f}±{np.std(r2s):.3f} | "
                  f"~{int(np.mean([x['n_features'] for x in sub]))} feats")

cv_df = pd.DataFrame(rows)
summary = (cv_df.groupby(["model", "layer", "feature_set"])
           .agg(r2_log_mean=("r2_log", "mean"), r2_log_std=("r2_log", "std"),
                rmse_log=("rmse_log", "mean"), mae_log=("mae_log", "mean"),
                r2_orig_mean=("r2_orig", "mean"), n_features=("n_features", "mean"))
           .reset_index().sort_values("r2_log_mean", ascending=False))
print("\n--- Summary (main_cv) ---")
print(summary.to_string(index=False))

best = summary.iloc[0]
B_MODEL, B_LAYER, B_FS = best["model"], best["layer"], best["feature_set"]
print(f"\nBest configuration: {B_MODEL} | {B_LAYER} | {B_FS} "
      f"(R²={best['r2_log_mean']:.3f})")

# ---------------------------------------------------------------------------
# 4. y-randomization on the best configuration
# ---------------------------------------------------------------------------
best_model = make_models()[B_MODEL]
r2_rand = []
for fid, (tr, va) in FOLDS.items():
    yp, _ = fit_predict(tr, va, LAYERS[B_LAYER], B_LAYER, B_FS, "y_rand",
                        B_MODEL, best_model)
    r2_rand.append(r2_score(va["y_rand"].values, yp))
verdict = "PASS ✓" if np.mean(r2_rand) < 0.05 else "FAIL ✗ — investigate leakage"
print(f"\ny-randomization ({B_MODEL}|{B_LAYER}|{B_FS}): "
      f"R²={np.mean(r2_rand):+.3f} → {verdict}")

# ---------------------------------------------------------------------------
# 5. Temporal hold-out (train on older papers, test on newest 20% of DOIs)
# ---------------------------------------------------------------------------
th = cv_splits[cv_splits["split_type"] == "temporal_holdout"]
tr_ids = set(th[th["role"] == "train"]["source_row_id"])
te_ids = set(th[th["role"] == "test"]["source_row_id"])
tr_t, te_t = df[df["source_row_id"].isin(tr_ids)], df[df["source_row_id"].isin(te_ids)]

temp_rows = []
print(f"\n--- Temporal hold-out: {len(tr_t)} train / {len(te_t)} test rows ---")
for layer in ["L1", "L3"]:
    for mname, model in make_models().items():
        yp, nfeat = fit_predict(tr_t, te_t, LAYERS[layer], layer, B_FS, "y",
                                mname, model)
        r = metrics(te_t["y"].values, yp)
        r.update({"layer": layer, "feature_set": B_FS, "model": mname,
                  "n_features": int(nfeat)})
        temp_rows.append(r)
        print(f"  {mname:9s} | {layer} | R²(log)={r['r2_log']:+.3f} | "
              f"RMSE={r['rmse_log']:.3f}")
temp_df = pd.DataFrame(temp_rows)

# ---------------------------------------------------------------------------
# 6. Plot: full vs selected per model × layer
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, layer in zip(axes, ["L1", "L2", "L3"]):
    s = summary[summary["layer"] == layer]
    models_ = sorted(s["model"].unique())
    x = np.arange(len(models_))
    for i, fs in enumerate(["full", "selected"]):
        ss = s[s["feature_set"] == fs].set_index("model").reindex(models_)
        ax.bar(x + (i - 0.5) * 0.38, ss["r2_log_mean"], 0.38,
               yerr=ss["r2_log_std"], capsize=3, label=fs)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(models_)
    ax.set_title(layer)
axes[0].set_ylabel("R² (log scale), GroupKFold by DOI")
axes[0].legend()
plt.tight_layout()
plt.savefig("/tmp/nb10_full_vs_selected.png", dpi=120)
plt.close()
print("Saved: /tmp/nb10_full_vs_selected.png")

# ---------------------------------------------------------------------------
# 7. Write Delta
# ---------------------------------------------------------------------------
spark.createDataFrame(cv_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.cv_results_nb10")
spark.createDataFrame(temp_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.temporal_results_nb10")
print(f"\n✓ Written: {SCHEMA}.cv_results_nb10 ({len(cv_df)} rows)")
print(f"✓ Written: {SCHEMA}.temporal_results_nb10 ({len(temp_df)} rows)")
print("\n=== NB10 COMPLETE ===")

In [0]:
# =============================================================================
# NB10 — Full vs selected feature sets, with in-fold selection + temporal test
# =============================================================================
# Objective:
#   1. Compare FULL vs SELECTED feature sets for L1 / L2 / L3 under the
#      serialized main_cv splits from NB07 (GroupKFold by DOI).
#      Feature selection is repeated INSIDE each fold (same procedure as NB09:
#      RF-MDI ranking -> |r|>0.90 correlation filter -> top-N -> physical
#      must-keeps), using only that fold's training rows. This avoids the
#      selection leakage that would occur if the global NB09 list were reused
#      (NB09 selected on fold-0 train, which contains val rows of folds 1-4).
#   2. y-randomization sanity check on the best configuration.
#   3. Temporal hold-out (most recent 20% of DOIs, from NB07) for the best
#      configuration: train on older papers, test on newer ones.
#
#   NOTE: RF OOB R² is NOT a valid metric here — bootstrap is row-level and
#   leaks intra-paper information. Only grouped CV / temporal hold-out are
#   reported.
#
# Inputs:
#   ml_perovskites.master, cv_splits, y_randomized,
#   X_material, X_thermo, X_exp
#
# Outputs:
#   ml_perovskites.cv_results_nb10      (fold × model × layer × feature_set)
#   ml_perovskites.temporal_results_nb10
#   /tmp/nb10_full_vs_selected.png
#
# Rules: imputation, target encoding and feature selection strictly inside
#        each fold. source_row_id / doi / doi_key never features. SEED = 42.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import copy
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute   import SimpleImputer
from sklearn.metrics  import r2_score, mean_squared_error, mean_absolute_error
from pyspark.sql import SparkSession

SEED        = 42
SCHEMA      = "ml_perovskites"
CORR_THRESH = 0.90
TOP_N       = {"L1": 40, "L2": 40, "L3": 50}
TE_M        = 10

# --- Run configuration ---
# v1 run: EXP_TABLE="X_exp",    RESULT_SUFFIX="",    LAYERS_TO_RUN=["L1","L2","L3"]
# v2 run: EXP_TABLE="X_exp_v2", RESULT_SUFFIX="_v2", LAYERS_TO_RUN=["L1","L3"]
#         (L2 dropped: ΔR²(L1→L2) ≈ 0 in v1 — thermo features redundant with L1)
EXP_TABLE     = "X_exp_v2"
RESULT_SUFFIX = "_v2"
LAYERS_TO_RUN = ["L1", "L3"]

CAT_COLS    = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type",
               "prep_class"]   # filtered below to those present in EXP_TABLE
MUST_KEEP_PATTERNS = [
    "goldschmidt", "octahedral", "Ra_Rb_ratio",
    "ECB_butler", "EVB_butler", "chi_mulliken",
    "co_catalyst", "scavenger", "run_type", "lamp_type",
    "catalyst_amount",
]

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

def try_import(pkg):
    try:
        return __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
        return __import__(pkg)

xgb = try_import("xgboost")
try:
    cb = try_import("catboost")
    HAS_CB = True
except Exception:
    HAS_CB = False
    print("WARNING: catboost unavailable — skipped.")

# ---------------------------------------------------------------------------
# 1. Load and align
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
cv_splits = spark.table(f"{SCHEMA}.cv_splits").toPandas()
y_rand_df = spark.table(f"{SCHEMA}.y_randomized").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.{EXP_TABLE}").toPandas()
CAT_COLS  = [c for c in CAT_COLS if c in X_exp_raw.columns]
print(f"Experimental table: {EXP_TABLE} | categoricals: {CAT_COLS}")

def is_use_h2(series):
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
df_base = master[
    is_use_h2(master["use_h2"]) &
    master["y_log10_1p_final"].notna() &
    master["doi_key"].notna()
].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

# Join key must be the same dtype everywhere
for t in [df_base, cv_splits, y_rand_df, X_mat_raw, X_thm_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META_COLS = {"source_row_id", "base_compound", "family_proposed",
             "formula_base_paper", "A_site_paper", "B_site_paper",
             "doi", "doi_key", "y_log10_1p_final", "y_log10_1p",
             "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}

for raw in [X_mat_raw, X_thm_raw]:
    for c in raw.columns:
        if c != "source_row_id":
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

def numeric_features(d):
    return [c for c in d.columns
            if c not in META_COLS and pd.api.types.is_numeric_dtype(d[c])]

x_mat_cols = numeric_features(X_mat_raw)
x_mat_cols = [c for c in x_mat_cols if X_mat_raw[c].isna().mean() < 0.50]
x_thm_cols = numeric_features(X_thm_raw)
x_thm_cols = [c for c in x_thm_cols if X_thm_raw[c].isna().mean() < 0.99]
x_exp_num  = [c for c in X_exp_raw.columns
              if c not in ["source_row_id"] + CAT_COLS]
for c in x_exp_num:
    X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")
x_exp_num  = [c for c in x_exp_num if X_exp_raw[c].notna().any()]

df = (df_base[["source_row_id", "doi_key", "y"]]
      .merge(X_mat_raw[["source_row_id"] + x_mat_cols], on="source_row_id", how="left")
      .merge(X_thm_raw[["source_row_id"] + x_thm_cols], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + x_exp_num + CAT_COLS],
             on="source_row_id", how="left"))
df = df.merge(y_rand_df[["source_row_id", "y_rand"]], on="source_row_id", how="left")
print(f"Merged shape: {df.shape}")

LAYERS = {
    "L1": x_mat_cols,
    "L2": x_mat_cols + x_thm_cols,
    "L3": x_mat_cols + x_thm_cols + x_exp_num + CAT_COLS,   # cats encoded in fold
}
LAYERS = {k: v for k, v in LAYERS.items() if k in LAYERS_TO_RUN}
for k, v in LAYERS.items():
    print(f"  {k}: {len(v)} candidate features")

# ---------------------------------------------------------------------------
# 2. In-fold helpers
# ---------------------------------------------------------------------------
def encode_fold(tr, te, cols, ycol):
    """Smoothed target encoding fitted on tr only; returns float frames."""
    Xtr, Xte = tr[cols].copy(), te[cols].copy()
    for cc in [c for c in CAT_COLS if c in cols]:
        gm = tr[ycol].mean()
        st = tr.groupby(cc)[ycol].agg(["mean", "count"])
        enc = ((st["mean"] * st["count"] + gm * TE_M) / (st["count"] + TE_M)).to_dict()
        Xtr[cc] = tr[cc].map(enc).fillna(gm)
        Xte[cc] = te[cc].map(enc).fillna(gm)
    return Xtr.astype(float), Xte.astype(float)

def select_in_fold(Xtr_df, ytr, layer):
    """NB09 procedure on training rows only: MDI rank -> corr filter -> top-N."""
    cols = list(Xtr_df.columns)
    X_imp = SimpleImputer(strategy="median").fit_transform(Xtr_df.values)
    rf = RandomForestRegressor(n_estimators=300, max_features="sqrt",
                               min_samples_leaf=3, n_jobs=-1, random_state=SEED)
    rf.fit(X_imp, ytr)
    ranked = [c for _, c in sorted(zip(rf.feature_importances_, cols), reverse=True)]
    corr = pd.DataFrame(X_imp, columns=cols).corr().abs()
    keep = []
    for f in ranked:
        if keep and corr.loc[f, keep].max() > CORR_THRESH:
            continue
        keep.append(f)
    top = set(ranked[:TOP_N[layer]])
    sel = [c for c in keep if c in top]
    must = [c for c in cols if any(k in c for k in MUST_KEEP_PATTERNS) and c not in sel]
    return list(dict.fromkeys(sel + must))

def make_models():
    m = {
        "RF": RandomForestRegressor(n_estimators=300, max_features="sqrt",
                                    min_samples_leaf=3, n_jobs=-1, random_state=SEED),
        "XGBoost": xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                                    subsample=0.8, colsample_bytree=0.7,
                                    min_child_weight=3, reg_lambda=1.0,
                                    tree_method="hist", random_state=SEED, verbosity=0),
    }
    if HAS_CB:
        m["CatBoost"] = cb.CatBoostRegressor(iterations=600, learning_rate=0.05,
                                             depth=6, l2_leaf_reg=3,
                                             random_seed=SEED, verbose=0)
    return m

def metrics(y_true, y_pred):
    yo_t, yo_p = 10 ** y_true - 1, 10 ** y_pred - 1
    return {
        "r2_log":   r2_score(y_true, y_pred),
        "rmse_log": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_log":  mean_absolute_error(y_true, y_pred),
        "r2_orig":  r2_score(yo_t, yo_p),
    }

def fit_predict(tr, te, cols, layer, feature_set, ycol, model_name, model):
    Xtr, Xte = encode_fold(tr, te, cols, ycol)
    if feature_set == "selected":
        sel = select_in_fold(Xtr, tr[ycol].values, layer)
        Xtr, Xte = Xtr[sel], Xte[sel]
    imp = SimpleImputer(strategy="median")
    Xtr_a = imp.fit_transform(Xtr.values)
    Xte_a = imp.transform(Xte.values)
    mdl = copy.deepcopy(model)
    mdl.fit(Xtr_a, tr[ycol].values)
    return mdl.predict(Xte_a), Xtr.shape[1]

# ---------------------------------------------------------------------------
# 3. Grouped CV: layer × feature_set × model (main_cv from NB07)
# ---------------------------------------------------------------------------
main_cv = cv_splits[cv_splits["split_type"] == "main_cv"]
fold_ids = sorted(main_cv["fold_id"].unique())

def fold_frames(fold_id):
    f = main_cv[main_cv["fold_id"] == fold_id]
    tr_ids = set(f[f["role"] == "train"]["source_row_id"])
    va_ids = set(f[f["role"] == "val"]["source_row_id"])
    return df[df["source_row_id"].isin(tr_ids)], df[df["source_row_id"].isin(va_ids)]

FOLDS = {fid: fold_frames(fid) for fid in fold_ids}

rows = []
print("\n--- Grouped CV (main_cv): full vs selected (in-fold) ---")
for layer, cols in LAYERS.items():
    for fs in ["full", "selected"]:
        for mname, model in make_models().items():
            for fid, (tr, va) in FOLDS.items():
                yp, nfeat = fit_predict(tr, va, cols, layer, fs, "y", mname, model)
                r = metrics(va["y"].values, yp)
                r.update({"layer": layer, "feature_set": fs, "model": mname,
                          "fold_id": int(fid), "n_features": int(nfeat),
                          "n_train": len(tr), "n_val": len(va)})
                rows.append(r)
            sub = [x for x in rows if x["layer"] == layer and
                   x["feature_set"] == fs and x["model"] == mname]
            r2s = [x["r2_log"] for x in sub]
            print(f"  {mname:9s} | {layer} | {fs:8s} | "
                  f"R²(log)={np.mean(r2s):+.3f}±{np.std(r2s):.3f} | "
                  f"~{int(np.mean([x['n_features'] for x in sub]))} feats")

cv_df = pd.DataFrame(rows)
summary = (cv_df.groupby(["model", "layer", "feature_set"])
           .agg(r2_log_mean=("r2_log", "mean"), r2_log_std=("r2_log", "std"),
                rmse_log=("rmse_log", "mean"), mae_log=("mae_log", "mean"),
                r2_orig_mean=("r2_orig", "mean"), n_features=("n_features", "mean"))
           .reset_index().sort_values("r2_log_mean", ascending=False))
print("\n--- Summary (main_cv) ---")
print(summary.to_string(index=False))

best = summary.iloc[0]
B_MODEL, B_LAYER, B_FS = best["model"], best["layer"], best["feature_set"]
print(f"\nBest configuration: {B_MODEL} | {B_LAYER} | {B_FS} "
      f"(R²={best['r2_log_mean']:.3f})")

# ---------------------------------------------------------------------------
# 4. y-randomization on the best configuration
# ---------------------------------------------------------------------------
best_model = make_models()[B_MODEL]
r2_rand = []
for fid, (tr, va) in FOLDS.items():
    yp, _ = fit_predict(tr, va, LAYERS[B_LAYER], B_LAYER, B_FS, "y_rand",
                        B_MODEL, best_model)
    r2_rand.append(r2_score(va["y_rand"].values, yp))
verdict = "PASS ✓" if np.mean(r2_rand) < 0.05 else "FAIL ✗ — investigate leakage"
print(f"\ny-randomization ({B_MODEL}|{B_LAYER}|{B_FS}): "
      f"R²={np.mean(r2_rand):+.3f} → {verdict}")

# ---------------------------------------------------------------------------
# 5. Temporal hold-out (train on older papers, test on newest 20% of DOIs)
# ---------------------------------------------------------------------------
th = cv_splits[cv_splits["split_type"] == "temporal_holdout"]
tr_ids = set(th[th["role"] == "train"]["source_row_id"])
te_ids = set(th[th["role"] == "test"]["source_row_id"])
tr_t, te_t = df[df["source_row_id"].isin(tr_ids)], df[df["source_row_id"].isin(te_ids)]

temp_rows = []
print(f"\n--- Temporal hold-out: {len(tr_t)} train / {len(te_t)} test rows ---")
for layer in ["L1", "L3"]:
    for mname, model in make_models().items():
        yp, nfeat = fit_predict(tr_t, te_t, LAYERS[layer], layer, B_FS, "y",
                                mname, model)
        r = metrics(te_t["y"].values, yp)
        r.update({"layer": layer, "feature_set": B_FS, "model": mname,
                  "n_features": int(nfeat)})
        temp_rows.append(r)
        print(f"  {mname:9s} | {layer} | R²(log)={r['r2_log']:+.3f} | "
              f"RMSE={r['rmse_log']:.3f}")
temp_df = pd.DataFrame(temp_rows)

# ---------------------------------------------------------------------------
# 6. Plot: full vs selected per model × layer
# ---------------------------------------------------------------------------
run_layers = list(LAYERS.keys())
fig, axes = plt.subplots(1, len(run_layers), figsize=(5 * len(run_layers), 4),
                         sharey=True, squeeze=False)
axes = axes[0]
for ax, layer in zip(axes, run_layers):
    s = summary[summary["layer"] == layer]
    models_ = sorted(s["model"].unique())
    x = np.arange(len(models_))
    for i, fs in enumerate(["full", "selected"]):
        ss = s[s["feature_set"] == fs].set_index("model").reindex(models_)
        ax.bar(x + (i - 0.5) * 0.38, ss["r2_log_mean"], 0.38,
               yerr=ss["r2_log_std"], capsize=3, label=fs)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(models_)
    ax.set_title(layer)
axes[0].set_ylabel("R² (log scale), GroupKFold by DOI")
axes[0].legend()
plt.tight_layout()
plot_path = f"/tmp/nb10_full_vs_selected{RESULT_SUFFIX}.png"
plt.savefig(plot_path, dpi=120)
plt.close()
print(f"Saved: {plot_path}")

# ---------------------------------------------------------------------------
# 7. Comparison with previous run (v1) if available
# ---------------------------------------------------------------------------
if RESULT_SUFFIX:
    try:
        prev = spark.table(f"{SCHEMA}.cv_results_nb10").toPandas()
        prev_s = (prev.groupby(["model", "layer", "feature_set"])["r2_log"]
                  .mean().rename("r2_v1"))
        cur_s  = (cv_df.groupby(["model", "layer", "feature_set"])["r2_log"]
                  .mean().rename("r2_v2"))
        comp = pd.concat([prev_s, cur_s], axis=1).dropna()
        comp["delta"] = comp["r2_v2"] - comp["r2_v1"]
        print("\n--- v1 (X_exp) vs v2 (X_exp_v2), grouped CV R²(log) ---")
        print(comp.round(3).to_string())
    except Exception as e:
        print(f"(no v1 table to compare: {e})")

# ---------------------------------------------------------------------------
# 8. Write Delta
# ---------------------------------------------------------------------------
cv_tbl   = f"{SCHEMA}.cv_results_nb10{RESULT_SUFFIX}"
temp_tbl = f"{SCHEMA}.temporal_results_nb10{RESULT_SUFFIX}"
spark.createDataFrame(cv_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(cv_tbl)
spark.createDataFrame(temp_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(temp_tbl)
print(f"\n✓ Written: {cv_tbl} ({len(cv_df)} rows)")
print(f"✓ Written: {temp_tbl} ({len(temp_df)} rows)")
print("\n=== NB10 COMPLETE ===")